# PoolGrad experiments

Builds and runs the PoolGrad Rust binary, parses stdout, and plots kernel/scheduler/pooling results.

# 1) Setup

In [ ]:
import os
import pathlib
import platform
import shutil
import subprocess


def _is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def _sh(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")


def _find_project_root(start: pathlib.Path) -> pathlib.Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "Cargo.toml").exists():
            return p
    raise FileNotFoundError(f"Could not find Cargo.toml from {start}")


def _guess_notebook_path() -> pathlib.Path | None:
    # VS Code Jupyter typically injects `__vsc_ipynb_file__` into the kernel globals.
    v = globals().get("__vsc_ipynb_file__")
    if isinstance(v, str) and v:
        p = pathlib.Path(v)
        if p.exists():
            return p
    # Some environments expose a full path via env vars.
    for key in ("VSCODE_NOTEBOOK_FILE", "VSCODE_IPYNB_FILE"):
        v = os.environ.get(key)
        if v:
            p = pathlib.Path(v)
            if p.exists():
                return p
    return None


is_colab = _is_colab()
notebook_path = _guess_notebook_path()
if notebook_path is not None:
    project_dir = _find_project_root(notebook_path.parent)
    project_dir_reason = f"notebook location: {notebook_path}"
else:
    project_dir = _find_project_root(pathlib.Path.cwd())
    project_dir_reason = "current working directory"

print("Platform:", platform.platform())
print("Initial CWD:", pathlib.Path.cwd())
print("Detected project_dir:", project_dir)
print("project_dir selected from:", project_dir_reason)

# Ensure Rust toolchain is available. On local macOS, we expect you to have Rust installed already.
if shutil.which("cargo") is None or shutil.which("rustc") is None:
    if not is_colab:
        raise RuntimeError(
            "Rust toolchain not found on PATH. Install from https://rustup.rs and restart the kernel. "
            "(Expected `cargo` and `rustc`.)"
        )

    # Colab path: install a recent stable toolchain via rustup (edition 2024 requires a new rustc).
    _sh("curl https://sh.rustup.rs -sSf | sh -s -- -y")
    os.environ["PATH"] = f"{pathlib.Path.home()}/.cargo/bin:" + os.environ.get("PATH", "")
    _sh("rustup toolchain install stable")
    _sh("rustup default stable")

_sh("rustc --version")
_sh("cargo --version")

In [ ]:
import sys
import subprocess


def _pip_install(packages: list[str]) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


try:
    import pandas as pd  # noqa: F401
    import matplotlib.pyplot as plt  # noqa: F401
except ModuleNotFoundError:
    _pip_install(["pandas", "matplotlib"])
    import pandas as pd  # noqa: F401
    import matplotlib.pyplot as plt  # noqa: F401

print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)

In [ ]:
# Demo configuration (tweak these for live demos)
BENCH_WARMUP = 2
BENCH_TRIALS = 9
MP_MAX_SIZE = 512

# If False and poolgrad_stdout.txt exists, reuse it instead of re-running cargo.
FORCE_RERUN = False

# Reduce notebook spam during demos (set >0 to print snippets).
# Recommended for talks/live demos: keep these at 0.
STDOUT_HEAD_LINES = 0
STDOUT_TAIL_LINES = 0

print("Config:")
print("  BENCH_WARMUP=", BENCH_WARMUP)
print("  BENCH_TRIALS=", BENCH_TRIALS)
print("  MP_MAX_SIZE=", MP_MAX_SIZE)
print("  FORCE_RERUN=", FORCE_RERUN)
print("  STDOUT_HEAD_LINES=", STDOUT_HEAD_LINES)
print("  STDOUT_TAIL_LINES=", STDOUT_TAIL_LINES)

In [ ]:
import os
import pathlib
import subprocess

# Reuse `project_dir` from the setup cell if present; otherwise, try to find Cargo.toml.
def _find_project_root(start: pathlib.Path) -> pathlib.Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "Cargo.toml").exists():
            return p
    raise FileNotFoundError(f"Could not find Cargo.toml from {start}")


project_dir = globals().get("project_dir")
if not isinstance(project_dir, pathlib.Path):
    project_dir = _find_project_root(pathlib.Path.cwd())

# Colab convenience: if this notebook is run standalone, clone the repo once.
is_colab = bool(globals().get("is_colab", False))
if is_colab and not (project_dir / "src").exists():
    repo_dir = pathlib.Path("poolgrad")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/Prigoistic/poolgrad.git"], check=True)
    project_dir = repo_dir.resolve()

os.chdir(project_dir)
print("CWD ->", pathlib.Path.cwd())

In [ ]:
!cargo build --release

# 2) Run benchmarks

The binary prints all experiment tables to stdout in one run.

In [ ]:
import os
import pathlib
import re
import subprocess

project_dir = pathlib.Path.cwd()
if not (project_dir / "Cargo.toml").exists():
    raise RuntimeError(f"Expected to run inside the PoolGrad project dir; Cargo.toml not found in {project_dir}")

# Read config from the "Demo configuration" cell if present; otherwise fall back to defaults.
BENCH_WARMUP = int(globals().get("BENCH_WARMUP", 2))
BENCH_TRIALS = int(globals().get("BENCH_TRIALS", 9))
MP_MAX_SIZE = int(globals().get("MP_MAX_SIZE", 512))
FORCE_RERUN = bool(globals().get("FORCE_RERUN", False))
STDOUT_HEAD_LINES = int(globals().get("STDOUT_HEAD_LINES", 0))
STDOUT_TAIL_LINES = int(globals().get("STDOUT_TAIL_LINES", 0))

stdout_path = project_dir / "poolgrad_stdout.txt"
if stdout_path.exists() and not FORCE_RERUN:
    raw = stdout_path.read_text()
    print(f"Reused existing {stdout_path.name} (bytes): {len(raw)}")
else:
    env = os.environ.copy()
    # Default to the project defaults, but set explicitly for reproducibility.
    env["POOLGRAD_BENCH_WARMUP"] = str(BENCH_WARMUP)
    env["POOLGRAD_BENCH_TRIALS"] = str(BENCH_TRIALS)
    env["POOLGRAD_MP_MAX_SIZE"] = str(MP_MAX_SIZE)

    result = subprocess.run(
        ["cargo", "run", "--release"],
        cwd=str(project_dir),
        env=env,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"poolgrad failed with exit code {result.returncode}")

    raw = result.stdout
    stdout_path.write_text(raw)
    print(f"Wrote {stdout_path.name} (bytes): {len(raw)}")

lines = raw.splitlines()
print("Captured stdout lines:", len(lines))

# Lightweight sanity summary: confirms the stdout format still matches what the parsers expect.
has_kernel_bench = any(l.startswith("Kernel benchmark") for l in lines)
has_step_bench = any(l.startswith("Forward+Backward step benchmark") for l in lines)
has_interaction = any(l.startswith("Kernel+Pool interaction experiment") for l in lines)
has_global_peak = any(l.startswith("Global peak memory") for l in lines)
pool_metric_re = re.compile(
    r"^(?P<mode>.*?\(pool (?:OFF|ON)\))\s*\|\s*(?P<alloc>\d+)\s*\|\s*(?P<reuse>\d+)\s*\|\s*(?P<reuse_rate>[0-9.]+)%\s*\|\s*(?P<live>\d+)\s*\|\s*(?P<cached>\d+)\s*\|\s*(?P<resident>\d+)\s*$"
)
pool_metric_lines = sum(1 for l in lines if pool_metric_re.match(l.strip()))
print(
    "Sections:",
    f"pool_metrics={pool_metric_lines}",
    f"kernel_bench={'yes' if has_kernel_bench else 'no'}",
    f"step_bench={'yes' if has_step_bench else 'no'}",
    f"interaction={'yes' if has_interaction else 'no'}",
    f"global_peak={'yes' if has_global_peak else 'no'}",
)

if STDOUT_HEAD_LINES > 0:
    print(f"--- stdout head (first {STDOUT_HEAD_LINES} lines) ---")
    print("\n".join(lines[:STDOUT_HEAD_LINES]))
if STDOUT_TAIL_LINES > 0:
    print(f"--- stdout tail (last {STDOUT_TAIL_LINES} lines) ---")
    print("\n".join(lines[-STDOUT_TAIL_LINES:]))

# 3) Parse output

Regex parsers below match the exact table formats printed by `src/main.rs`.

In [ ]:
import pathlib
import re

import pandas as pd

# Allow running this cell out-of-order by loading previously captured stdout.
if "raw" not in globals() or not isinstance(raw, str) or not raw.strip():
    stdout_path = pathlib.Path("poolgrad_stdout.txt")
    if stdout_path.exists():
        raw = stdout_path.read_text()
    else:
        raise RuntimeError(
            "No benchmark stdout found. Run the benchmarks cell first, "
            "or ensure poolgrad_stdout.txt exists in the project directory."
        )


def parse_pool_metrics(raw: str) -> pd.DataFrame:
    # Matches lines produced by print_metrics() in src/main.rs
    metric_re = re.compile(
        r"^(?P<mode>.*?\(pool (?:OFF|ON)\))\s*\|\s*(?P<alloc>\d+)\s*\|\s*(?P<reuse>\d+)\s*\|\s*(?P<reuse_rate>[0-9.]+)%\s*\|\s*(?P<live>\d+)\s*\|\s*(?P<cached>\d+)\s*\|\s*(?P<resident>\d+)\s*$"
    )

    rows = []
    for line in raw.splitlines():
        m = metric_re.match(line.strip())
        if not m:
            continue
        rows.append(
            {
                "mode": m.group("mode"),
                "allocations": int(m.group("alloc")),
                "reuses": int(m.group("reuse")),
                "reuse_rate_pct": float(m.group("reuse_rate")),
                "live_peak_bytes": int(m.group("live")),
                "cached_peak_bytes": int(m.group("cached")),
                "resident_peak_bytes": int(m.group("resident")),
            }
        )

    if not rows:
        raise ValueError("Failed to parse pooling metrics: no matching lines found.")

    return pd.DataFrame(rows)


def parse_kernel_benchmark(raw: str) -> pd.DataFrame:
    lines = raw.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.startswith("Kernel benchmark"):
            start = i
            break
    if start is None:
        raise ValueError("Failed to find Kernel benchmark section header in stdout.")

    rows = []
    for line in lines[start + 1 :]:
        if line.startswith("Forward+Backward step benchmark") or line.startswith(
            "Kernel+Pool interaction experiment"
        ):
            break
        if not re.match(r"^\s*\d+\s*\|", line):
            continue

        parts = [p.strip() for p in line.split("|")]
        parts = [p for p in parts if p != ""]
        if len(parts) < 11:
            continue

        rows.append(
            {
                "size": int(parts[0]),
                "naive_med_ms": float(parts[1]),
                "naive_p95_ms": float(parts[2]),
                "tiled_med_ms": float(parts[3]),
                "tiled_p95_ms": float(parts[4]),
                "packed_med_ms": float(parts[5]),
                "packed_p95_ms": float(parts[6]),
                "mp_med_ms": float(parts[7]),
                "mp_p95_ms": float(parts[8]),
                "winner": parts[9],
                "scheduler": parts[10],
            }
        )

    if not rows:
        raise ValueError("Failed to parse Kernel benchmark table: no data rows found.")

    return pd.DataFrame(rows).sort_values("size")


def parse_interaction_experiment(raw: str) -> pd.DataFrame:
    lines = raw.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.startswith("Kernel+Pool interaction experiment"):
            start = i
            break
    if start is None:
        raise ValueError(
            "Failed to find Kernel+Pool interaction experiment section header in stdout."
        )

    row_re = re.compile(
        r"^\s*(?P<size>\d+)\s*\|\s*(?P<kernel>[A-Za-z]+)\s*\|\s*(?P<iters>\d+)\s*\|\s*(?P<time>[0-9.]+)\s*\|\s*(?P<alloc>\d+)\s*\|\s*(?P<reuse>\d+)\s*\|\s*(?P<live>\d+)\s*\|\s*(?P<resident>\d+)\s*$"
    )

    rows = []
    for line in lines[start + 1 :]:
        if line.startswith("Global peak memory"):
            break
        m = row_re.match(line)
        if not m:
            continue
        rows.append(
            {
                "size": int(m.group("size")),
                "kernel": m.group("kernel"),
                "iters": int(m.group("iters")),
                "time_ms": float(m.group("time")),
                "alloc": int(m.group("alloc")),
                "reuse": int(m.group("reuse")),
                "live_peak_bytes": int(m.group("live")),
                "resident_peak_bytes": int(m.group("resident")),
            }
        )

    if not rows:
        raise ValueError(
            "Failed to parse Kernel+Pool interaction experiment table: no data rows found."
        )

    return pd.DataFrame(rows).sort_values("size")


pool_df = parse_pool_metrics(raw)
kernel_df = parse_kernel_benchmark(raw)
interaction_df = parse_interaction_experiment(raw)

display(pool_df)
display(kernel_df)
display(interaction_df)

# 4) Kernel performance

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(kernel_df['size'], kernel_df['naive_med_ms'], marker='o', label='Naive')
plt.plot(kernel_df['size'], kernel_df['tiled_med_ms'], marker='o', label='Tiled')
plt.plot(kernel_df['size'], kernel_df['packed_med_ms'], marker='o', label='Packed+SIMD')
plt.plot(kernel_df['size'], kernel_df['mp_med_ms'], marker='o', label='MP')

plt.title('Kernel runtime (median)')
plt.xlabel('Matrix size (N, square GEMM)')
plt.ylabel('Runtime (ms)')
plt.legend()
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

# 5) Speedup vs Naive

In [ ]:
speedup_df = pd.DataFrame({
    'size': kernel_df['size'],
    'Tiled': kernel_df['naive_med_ms'] / kernel_df['tiled_med_ms'],
    'Packed+SIMD': kernel_df['naive_med_ms'] / kernel_df['packed_med_ms'],
    'MP': kernel_df['naive_med_ms'] / kernel_df['mp_med_ms'],
})

plt.figure()
plt.plot(speedup_df['size'], speedup_df['Tiled'], marker='o', label='Tiled')
plt.plot(speedup_df['size'], speedup_df['Packed+SIMD'], marker='o', linewidth=2.5, label='Packed+SIMD')
plt.plot(speedup_df['size'], speedup_df['MP'], marker='o', label='MP')

plt.title('Speedup vs Naive (median)')
plt.xlabel('Matrix size (N, square GEMM)')
plt.ylabel('Speedup (naive_time / kernel_time)')
plt.legend()
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

# 6) Memory pooling

In [ ]:
# Bar chart: allocations and reuses, baseline vs pooled
labels = pool_df['mode'].tolist()
x = list(range(len(labels)))

alloc = pool_df['allocations'].tolist()
reuse = pool_df['reuses'].tolist()

width = 0.35
plt.figure()
plt.bar([i - width / 2 for i in x], alloc, width, label='Allocations')
plt.bar([i + width / 2 for i in x], reuse, width, label='Reuses')

plt.title('Pooling behavior (training loop)')
plt.xlabel('Mode')
plt.ylabel('Count')
plt.xticks(x, labels, rotation=15, ha='right')
plt.legend()
plt.grid(True, axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

# 7) Kernel + pool interaction

Plots total time for the combined forward matmul + seeded backward loop, and annotates reuse counts.

In [ ]:
plt.figure()
plt.plot(interaction_df['size'], interaction_df['time_ms'], marker='o')

for _, row in interaction_df.iterrows():
    plt.annotate(
        f"reuse={int(row['reuse'])}",
        (row['size'], row['time_ms']),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
    )

plt.title('Kernel + pool interaction (scheduler-selected kernel)')
plt.xlabel('Matrix size (N)')
plt.ylabel('Total time (ms)')
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

# 8) Scheduler behavior (optional)

Compares the scheduler choice to the fastest kernel observed in the kernel benchmark table.

In [ ]:
def normalize_kernel_name(name: str) -> str:
    # Winner uses short labels; scheduler uses enum debug names.
    if name == 'TiledPk':
        return 'TiledPacked'
    return name

sched_df = kernel_df[['size', 'winner', 'scheduler']].copy()
sched_df['winner_norm'] = sched_df['winner'].map(normalize_kernel_name)
sched_df['scheduler_norm'] = sched_df['scheduler'].map(normalize_kernel_name)
sched_df['match'] = sched_df['winner_norm'] == sched_df['scheduler_norm']

display(sched_df[['size', 'winner_norm', 'scheduler_norm', 'match']])

kernel_order = ['Naive', 'Tiled', 'TiledPacked', 'TiledMP']
y = {k: i for i, k in enumerate(kernel_order)}

plt.figure()
plt.scatter(sched_df['size'], sched_df['winner_norm'].map(y), label='Fastest (observed)', marker='x')
plt.scatter(sched_df['size'], sched_df['scheduler_norm'].map(y), label='Scheduler choice', marker='o')
plt.yticks(list(range(len(kernel_order))), kernel_order)
plt.title('Scheduler choice vs fastest observed')
plt.xlabel('Matrix size (N)')
plt.ylabel('Kernel')
plt.legend()
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
plt.show()

# 9) Observations (derived, short)

In [ ]:
# Derive a few short, run-specific statements (no hardcoded claims).
max_n = int(kernel_df['size'].max())
row_max = kernel_df[kernel_df['size'] == max_n].iloc[0]

alloc_off = int(pool_df[pool_df['mode'].str.contains('OFF')]['allocations'].iloc[0])
alloc_on = int(pool_df[pool_df['mode'].str.contains('ON')]['allocations'].iloc[0])
reuse_off = int(pool_df[pool_df['mode'].str.contains('OFF')]['reuses'].iloc[0])
reuse_on = int(pool_df[pool_df['mode'].str.contains('ON')]['reuses'].iloc[0])

print(f'- Fastest kernel at N={max_n} (median): {row_max.winner}')
print(f"- Packed+SIMD speedup at N={max_n}: {row_max.naive_med_ms / row_max.packed_med_ms:.2f}x vs naive")
print(f'- Pool allocations: {alloc_off} -> {alloc_on}')
print(f'- Pool reuses: {reuse_off} -> {reuse_on}')

mismatches = int((kernel_df['winner'].map(normalize_kernel_name) != kernel_df['scheduler'].map(normalize_kernel_name)).sum())
print(f'- Scheduler mismatches (winner != scheduled) across tested sizes: {mismatches} / {len(kernel_df)}')
